# Embeddings

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=140, precision=4, suppress=True)
torch.manual_seed(42)

%matplotlib inline

### Categorical variables in neural networks

Structured datasets often contain **categorical variables** -- variables that take one of a finite set of discrete values. Examples include a store ID, the day of the week, or the German state a store is located in.

Neural networks operate on real-valued vectors, so we need to convert these categories into numbers. There are two natural approaches:

1. **Integer encoding**: assign each category an integer ($0, 1, 2, \ldots$). Simple, but this imposes a fake ordering -- the network will treat "store 500" as halfway between "store 1" and "store 1000", which is meaningless.

2. **One-hot encoding**: represent each category as a binary vector with a 1 in one position and 0s elsewhere. This eliminates the fake ordering, but creates two new problems:
   - **High dimensionality**: a variable with $k$ values needs a $k$-dimensional vector. With 1,115 stores, the store ID alone requires a 1,115-dimensional input.
   - **No structure**: every pair of categories is placed at the same distance from every other, so the representation carries no information about which categories behave similarly.

In [ ]:
# One-hot encode 7 days of the week
n_days  = 7
one_hot = np.eye(n_days)

dist = lambda a, b: np.linalg.norm(one_hot[a] - one_hot[b])

print(f'Distance (Monday, Tuesday):   {dist(0, 1):.4f}')
print(f'Distance (Monday, Wednesday): {dist(0, 2):.4f}')
print(f'Distance (Monday, Sunday):    {dist(0, 6):.4f}')

Every pairwise distance is $\sqrt{2}$. The representation carries no information about which days are adjacent, which are weekdays, or anything else about the structure of the variable.

**Entity embeddings** offer a third approach: *learn* a dense, low-dimensional vector for each category directly from the data. Instead of a fixed look-up table of one-hot vectors, we learn a weight matrix

$$\mathbf{W} \in \mathbb{R}^{k \times d}$$

where $k$ is the number of categories and $d \ll k$ is the **embedding dimension**. The embedding for category $i$ is the $i$-th row: $\mathbf{e}_i = \mathbf{W}[i, :]$.

This is exactly what `nn.Embedding` does in PyTorch -- a differentiable look-up table updated by backpropagation. Categories that have similar effects on the output end up close to each other in the embedding space.

We will compare both approaches on the same dataset.

### The Rossmann dataset

The [Kaggle Rossmann Store Sales competition](https://www.kaggle.com/c/rossmann-store-sales) (2015) asked competitors to predict daily sales for 1,115 Rossmann drug stores across Germany.

The features we use:

| Feature | Type | # Values |
|---|---|---|
| Store | nominal | 1,115 |
| Day of week | ordinal | 7 |
| Month | ordinal | 12 |
| Year | ordinal | 3 (2013--2015) |
| Promo | binary | 2 |
| State | nominal | 12 |

The target is daily sales. We predict $\log(\text{Sales})$ normalized to $[0, 1]$.

In [ ]:
train        = pd.read_csv('train.csv', low_memory=False)
store_states = pd.read_csv('store_states.csv')

df = train.merge(store_states, on='Store')
df = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()

print(f'Rows after filtering: {len(df):,}')
df.head()

In [ ]:
# Temporal features (0-indexed)
df['Date']      = pd.to_datetime(df['Date'])
df['Month']     = df['Date'].dt.month - 1        # 0..11
df['YearIdx']   = df['Date'].dt.year  - 2013     # 0, 1, 2
df['DayOfWeek'] = df['DayOfWeek'] - 1            # 0..6
df['StoreIdx']  = df['Store'] - 1               # 0..1114

# State integer codes
states_sorted  = sorted(df['State'].unique())
state_to_idx   = {s: i for i, s in enumerate(states_sorted)}
df['StateIdx'] = df['State'].map(state_to_idx)

# Target: log(Sales) normalized to [0, 1]
df['log_sales'] = np.log(df['Sales'])
log_sales_max   = df['log_sales'].max()
df['target']    = df['log_sales'] / log_sales_max

n_stores = df['StoreIdx'].nunique()
n_states = len(states_sorted)
n_ohe    = n_stores + 7 + 12 + 3 + n_states + 1   # one-hot input dimension

print(f'Stores: {n_stores},  States: {n_states}')
print(f'One-hot input dimension: {n_ohe}')

In [ ]:
# Random 80/20 split (same stores in train and val, drawn from the same time period)
df_train_rnd = df.sample(frac=0.8, random_state=0)
df_val_rnd   = df.drop(df_train_rnd.index).reset_index(drop=True)
df_train_rnd = df_train_rnd.reset_index(drop=True)

# Temporal split: train on 2013-2014, validate on 2015
df_train_tmp = df[df['Date'] <  '2015-01-01'].reset_index(drop=True)
df_val_tmp   = df[df['Date'] >= '2015-01-01'].reset_index(drop=True)

print(f'Random   -- Train: {len(df_train_rnd):,}   Val: {len(df_val_rnd):,}')
print(f'Temporal -- Train: {len(df_train_tmp):,}   Val: {len(df_val_tmp):,}')

### Dataset and training utilities

The `RossmannDataset` converts a DataFrame into integer-index tensors that both models will consume. The one-hot model converts those indices to binary vectors inside its `forward` method; the embedding model looks them up in learned weight matrices.

In [ ]:
class RossmannDataset(Dataset):
    def __init__(self, df):
        self.store  = torch.tensor(df['StoreIdx'].values,  dtype=torch.long)
        self.dow    = torch.tensor(df['DayOfWeek'].values, dtype=torch.long)
        self.month  = torch.tensor(df['Month'].values,     dtype=torch.long)
        self.year   = torch.tensor(df['YearIdx'].values,   dtype=torch.long)
        self.state  = torch.tensor(df['StateIdx'].values,  dtype=torch.long)
        self.promo  = torch.tensor(df['Promo'].values,     dtype=torch.float)
        self.target = torch.tensor(df['target'].values,    dtype=torch.float)

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):
        return (
            self.store[idx], self.dow[idx], self.month[idx],
            self.year[idx],  self.state[idx], self.promo[idx],
            self.target[idx]
        )


def make_loaders(df_train, df_val):
    train_loader = DataLoader(RossmannDataset(df_train), batch_size=1024, shuffle=True)
    val_loader   = DataLoader(RossmannDataset(df_val),   batch_size=2048, shuffle=False)
    return train_loader, val_loader


def train_model(model, loader, n_epochs=10, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    losses    = []
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        for batch in loader:
            store, dow, month, year, state, promo, target = batch
            pred = model(store, dow, month, year, state, promo)
            loss = loss_fn(pred, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
    return losses


def eval_mape(model, loader):
    """Mean absolute percentage error on the original sales scale."""
    model.eval()
    ape_sum, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            store, dow, month, year, state, promo, target = batch
            pred_sales = torch.exp(model(store, dow, month, year, state, promo) * log_sales_max)
            true_sales = torch.exp(target * log_sales_max)
            ape_sum   += ((pred_sales - true_sales).abs() / true_sales).sum().item()
            n         += len(target)
    return ape_sum / n


train_loader_rnd, val_loader_rnd = make_loaders(df_train_rnd, df_val_rnd)

### Approach 1: One-hot encoding

Each categorical variable is expanded into a one-hot vector inside the network's `forward` method. The resulting vectors are concatenated with the continuous Promo feature to form the input to the first dense layer.

The input dimension is $1{,}115 + 7 + 12 + 3 + 12 + 1 = 1{,}150$.

In [ ]:
class OneHotNet(nn.Module):
    def __init__(self, n_stores, n_states):
        super().__init__()
        self.n_stores = n_stores
        self.n_states = n_states
        n_in = n_stores + 7 + 12 + 3 + n_states + 1
        self.fc = nn.Sequential(
            nn.Linear(n_in, 200),
            nn.ReLU(),
            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Linear(100, 1),
            nn.Sigmoid(),
        )

    def forward(self, store, dow, month, year, state, promo):
        x = torch.cat([
            F.one_hot(store, self.n_stores).float(),
            F.one_hot(dow,   7).float(),
            F.one_hot(month, 12).float(),
            F.one_hot(year,  3).float(),
            F.one_hot(state, self.n_states).float(),
            promo.unsqueeze(1),
        ], dim=1)
        return self.fc(x).squeeze(1)


ohe_model   = OneHotNet(n_stores=n_stores, n_states=n_states)
ohe_nparams = sum(p.numel() for p in ohe_model.parameters())
print(f'One-hot model parameters: {ohe_nparams:,}')

In [ ]:
print('Training one-hot model (random split)...')
ohe_losses_rnd = train_model(ohe_model, train_loader_rnd, n_epochs=10)

ohe_mape_rnd = eval_mape(ohe_model, val_loader_rnd)
print(f'\nOne-hot validation MAPE (random): {ohe_mape_rnd:.4f}')

### Approach 2: Entity embeddings

Instead of a fixed one-hot vector, each categorical variable is mapped to a learned dense vector. The embedding for category $i$ is the $i$-th row of a weight matrix $\mathbf{W} \in \mathbb{R}^{k \times d}$ that is updated during backpropagation like any other weight.

The embedding dimensions follow the paper:

| Feature | # Values | Embedding dim |
|---|---|---|
| Store | 1,115 | 10 |
| Day of week | 7 | 4 |
| Month | 12 | 4 |
| Year | 3 | 2 |
| State | 12 | 6 |

The concatenated embedding vector fed into the first dense layer has dimension $10 + 4 + 4 + 2 + 6 + 1 = 27$, compared to $1{,}150$ for the one-hot model. The fully-connected layers are otherwise identical.

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, n_stores, n_states):
        super().__init__()
        self.emb_store = nn.Embedding(n_stores, 10)
        self.emb_dow   = nn.Embedding(7,         4)
        self.emb_month = nn.Embedding(12,        4)
        self.emb_year  = nn.Embedding(3,         2)
        self.emb_state = nn.Embedding(n_states,  6)

        n_in = 10 + 4 + 4 + 2 + 6 + 1
        self.fc = nn.Sequential(
            nn.Linear(n_in, 200),
            nn.ReLU(),
            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Linear(100, 1),
            nn.Sigmoid(),
        )

    def forward(self, store, dow, month, year, state, promo):
        x = torch.cat([
            self.emb_store(store),
            self.emb_dow(dow),
            self.emb_month(month),
            self.emb_year(year),
            self.emb_state(state),
            promo.unsqueeze(1),
        ], dim=1)
        return self.fc(x).squeeze(1)


emb_model   = EmbeddingNet(n_stores=n_stores, n_states=n_states)
emb_nparams = sum(p.numel() for p in emb_model.parameters())
print(f'Embedding model parameters: {emb_nparams:,}')

In [ ]:
print('Training embedding model (random split)...')
emb_losses_rnd = train_model(emb_model, train_loader_rnd, n_epochs=10)

emb_mape_rnd = eval_mape(emb_model, val_loader_rnd)
print(f'\nEmbedding validation MAPE (random): {emb_mape_rnd:.4f}')

### Comparison: random split

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, 11)
axes[0].plot(epochs, ohe_losses_rnd, marker='o', label='One-hot')
axes[0].plot(epochs, emb_losses_rnd, marker='s', label='Embeddings')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train MSE (normalized log-sales)')
axes[0].set_title('Training loss (random split)')
axes[0].legend()

axes[1].bar(['One-hot', 'Embeddings'], [ohe_mape_rnd, emb_mape_rnd], color=['steelblue', 'darkorange'])
axes[1].set_ylabel('Validation MAPE')
axes[1].set_title('Validation MAPE (random split)')
for i, v in enumerate([ohe_mape_rnd, emb_mape_rnd]):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f'Parameters -- one-hot: {ohe_nparams:,}   embeddings: {emb_nparams:,}')

### Why does OHE win on the random split?

With a random 80/20 split, the **same 1,115 stores appear in both train and val**. The two sets differ only in which particular days are held out. This means the OHE model can memorize the baseline sales level of every store and apply it exactly at validation time -- and there are ~600 observations per store to learn from, so those memorized baselines are reliable.

The embedding model imposes a bottleneck: it must represent each store in 10 numbers. That prevents the kind of direct memorization OHE does, which is a disadvantage when memorization is exactly what works.

### Temporal split: a more realistic evaluation

A random split is not how a sales forecasting model would actually be evaluated. In practice you train on historical data and forecast the future. With a **temporal split** (train on 2013--2014, validate on 2015) the model must genuinely generalize -- it cannot simply recall patterns from randomly held-out days of the same time period.

Here, OHE's large parameter count (\~250k) becomes a liability: with 250k parameters fit on two years of training data, the model overfits to the idiosyncratic patterns of 2013-2014, and those learned patterns do not transfer as cleanly to 2015. The embedding model's tighter representation (\~37k parameters) learns more compact, transferable structure.

In [ ]:
train_loader_tmp, val_loader_tmp = make_loaders(df_train_tmp, df_val_tmp)

ohe_tmp = OneHotNet(n_stores=n_stores, n_states=n_states)
emb_tmp = EmbeddingNet(n_stores=n_stores, n_states=n_states)

print('Training one-hot model (temporal split)...')
train_model(ohe_tmp, train_loader_tmp, n_epochs=10)
ohe_mape_tmp = eval_mape(ohe_tmp, val_loader_tmp)

print('Training embedding model (temporal split)...')
train_model(emb_tmp, train_loader_tmp, n_epochs=10)
emb_mape_tmp = eval_mape(emb_tmp, val_loader_tmp)

print(f'\nOne-hot   val MAPE (temporal): {ohe_mape_tmp:.4f}')
print(f'Embedding val MAPE (temporal): {emb_mape_tmp:.4f}')

In [ ]:
emb_mape_tmp

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

x      = np.arange(2)
width  = 0.35
labels = ['One-hot', 'Embeddings']
rnd    = [ohe_mape_rnd, emb_mape_rnd]
tmp    = [ohe_mape_tmp, emb_mape_tmp]

bars1 = ax.bar(x - width/2, rnd, width, label='Random split',   color=['steelblue', 'darkorange'])
bars2 = ax.bar(x + width/2, tmp, width, label='Temporal split', color=['steelblue', 'darkorange'], alpha=0.5)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Validation MAPE')
ax.set_title('OHE vs Embeddings: random vs temporal split')
ax.legend()
plt.tight_layout()
plt.show()

### Visualizing the learned embeddings

After training, the rows of each embedding weight matrix encode learned properties of the corresponding categories. We visualize the **state embeddings** by projecting onto the first two principal components.

Guo & Berkhahn (2016) find that these 2D embeddings roughly reproduce the geographic layout of Germany -- eastern states cluster together, western states cluster together, and city-states (Berlin, Hamburg) appear as outliers. This geographic structure emerges entirely from patterns in sales data, without the network ever receiving coordinate information.

In [ ]:
state_name_map = {
    'BE':    'Berlin',
    'BW':    'Baden-Wuerttemberg',
    'BY':    'Bavaria',
    'HB,NI': 'Bremen/Lower Saxony',
    'HE':    'Hesse',
    'HH':    'Hamburg',
    'NW':    'North Rhine-Westphalia',
    'RP':    'Rhineland-Palatinate',
    'SH':    'Schleswig-Holstein',
    'SN':    'Saxony',
    'ST':    'Saxony-Anhalt',
    'TH':    'Thuringia',
}

state_emb = emb_model.emb_state.weight.detach().numpy()   # (n_states, 6)
pca       = PCA(n_components=2)
state_2d  = pca.fit_transform(state_emb)

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(state_2d[:, 0], state_2d[:, 1], s=80, color='steelblue')

for i, abbr in enumerate(states_sorted):
    ax.annotate(
        state_name_map.get(abbr, abbr),
        xy=(state_2d[i, 0], state_2d[i, 1]),
        xytext=(6, 4), textcoords='offset points', fontsize=9
    )

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel(f'PC 1  ({pca.explained_variance_ratio_[0]:.1%} of variance)')
ax.set_ylabel(f'PC 2  ({pca.explained_variance_ratio_[1]:.1%} of variance)')
ax.set_title('German State Embeddings (PCA to 2D)')
plt.tight_layout()
plt.show()

### Day-of-week embeddings

We can apply the same visualization to the day-of-week embeddings. What weekly structure, if any, did the network discover?

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

dow_emb  = emb_model.emb_dow.weight.detach().numpy()   # (7, 4)
pca_dow  = PCA(n_components=2)
dow_2d   = pca_dow.fit_transform(dow_emb)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(dow_2d[:, 0], dow_2d[:, 1], s=100, color='steelblue')

for i, name in enumerate(day_names):
    ax.annotate(
        name, xy=(dow_2d[i, 0], dow_2d[i, 1]),
        xytext=(6, 4), textcoords='offset points', fontsize=10
    )

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel(f'PC 1  ({pca_dow.explained_variance_ratio_[0]:.1%} of variance)')
ax.set_ylabel(f'PC 2  ({pca_dow.explained_variance_ratio_[1]:.1%} of variance)')
ax.set_title('Day-of-Week Embeddings (PCA to 2D)')
plt.tight_layout()
plt.show()

## References

* Guo, Cheng, and Felix Berkhahn. "Entity Embeddings of Categorical Variables." arXiv:1604.06737 (2016).